In [1]:
import tensorflow as tf
gpu_devices = tf.config.experimental.list_physical_devices('GPU')
device = gpu_devices[0]
tf.config.experimental.set_memory_growth(device, True)
import tensorflow_probability as tfp
tfd = tfp.distributions
tfb = tfp.bijectors
import numpy as np
import matplotlib.pyplot as plt
import scipy.io as sio
import sys
from classes.objects import *
from vine_tree.tree_op import *

from scipy import stats
import pickle

###########

from param.generate_rvine import *
from param.margin_fit import *
from param.margin_op import *
from param.copula_fit import *
from param.cond_copula import *
from pre_proc.preparation import prep_cop
from pred.prediction import*
from sampling.vine_sample import *
from info.info_estimation import vine_entropy

In [2]:
#### Generate random matrix

cases = 1000        ### Number of samples
vine_type = 'c-vine' # or 'd-vine' or 'c-vine'
method = 'matrix'  # or 'r_matrix'  only with r-vine
binning = False
n_bin = 3
dim = 10                # Dimension of the vine for random r-vine or c-vine or d-vine


if vine_type == 'r-vine':
    
    if method == 'matrix':
        
        ######### REGULAR MATRIX
        r_matrix = np.array([[2, 0, 0, 0, 0],
                             [5, 3, 0, 0, 0],
                             [4, 5, 1, 0, 0],
                             [1, 4, 5, 4, 0],
                             [3, 1, 4, 5, 5]])
        
#         r_matrix = np.array([[3, 0, 0, 0],
#                              [1, 4, 0, 0],
#                              [2, 1, 2, 0],
#                              [4, 2, 1, 1]])
        
#         r_matrix = np.array([[3, 0, 0],
#                              [2, 2, 0],
#                              [1, 1, 1]])

        print(r_matrix)
        
    elif method == 'random':
        
        ##### RANDOM R-MATRIX
        r_matrix, ind_vine, nodes, E = random_r_matrix_gen(dim)
        print(r_matrix)

    E, ind_vine, nodes, matrix_edges = prepare_regular(r_matrix)
    print('matrix_edges',matrix_edges)
    
    ## DEFINE MARGINS
#     margin_fam1 = ['norm','gamma','norm','gamma','norm']
#     theta_fam1 = [[0,1],[2,4],[0,1],[2,4],[0,1]]
    margin_fam1 = ['norm','norm','norm','norm','norm']
    theta_fam1 = [[0,1],[0,1],[0,1],[0,1],[0,1]]
    is_cont1 = [True,True,True,True,True]

    margin_vine = []
    for i in range(0,len(margin_fam1),1):
        mar_p = margin_obj(margin_fam1[i], theta_fam1[i], is_cont1[i])
        margin_vine.append(mar_p)

    for i in range(0,len(margin_fam1),1):
        print(margin_vine[i].dist, end =' ')
        print(margin_vine[i].theta, end =' ')
    
    ######################## DEFINE COPULAS  ###################
    if not binning:
        
#         margin_cop1 = [['gaussian','gaussian','gaussian','gaussian'],
#                        ['gaussian','gaussian','gaussian'],
#                        ['gaussian','gaussian'],
#                        ['gaussian']]
        
#         theta_cop1 = [[0.3, 0.5, 0.7, 0.8],
#                       [0.5, 0.8, 0.4],
#                       [0.5, 0.3],
#                       [0.9]]

#         margin_cop1 = [['gaussian','gaussian','gaussian','gaussian','gaussian'],
#                        ['gaussian','gaussian','gaussian','gaussian'],
#                        ['gaussian','gaussian','gaussian'],
#                        ['gaussian','gaussian'],
#                        ['gaussian']]
        
#         theta_cop1 = [[0.3, 0.5, 0.7, 0.8, -0.8],
#                       [0.5, 0.8, 0.4, -0.2],
#                       [0.5, 0.3, -0.7],
#                       [0.9, 0.6],
#                       [0.5]]
        
        margin_cop1 = [['gaussian','student','clayton','gaussian'],
                       ['student','clayton','gaussian'],
                       ['student','gaussian'],
                       ['clayton']]

        theta_cop1 = [[0.3, [0,0.2], 0.7,  -0.8],
                      [[-0.8,0.2], 4.5,  -0.2],
                      [[0,0.5],  -0.7],
                      [0.9]]
        
#         margin_cop1 = [['gaussian','gaussian','gaussian'],
#                        ['gaussian','gaussian'],
#                        ['gaussian']]

#         theta_cop1 = [[0.7, 0.8, 0.9],
#                       [0.6, 0.5],
#                       [0.7]]

        # margin_cop1 = [['clayton','clayton','clayton'],
        #                ['clayton','clayton'],
        #                ['clayton']]

        # theta_cop1 = [[3.7, 4.8, 5.9],
        #               [6.6, 2.5],
        #               [5.7]]
    else:
        
        margin_cop1 = [['gaussian','gaussian','gaussian','gaussian'],
                       [['gaussian','gaussian','gaussian'],['gaussian','gaussian','gaussian'],['gaussian','gaussian','gaussian']],
                      [['gaussian','gaussian','gaussian'],['gaussian','gaussian','gaussian']],
                      [['gaussian','gaussian','gaussian']]]

        theta_cop1 = [[0.3, 0.5, 0.7, 0.8],
                      [[0.3, 0.4, 0.5],[0.6, 0.7, 0.8],[0.2, 0.3, 0.4]],
                      [[0.3, 0.4, 0.5],[0.3, 0.5, 0.9]],
                      [[0.2, 0.5, 0.9]]] #0.2,0.5,0.9
        
#         margin_cop1 = [['gaussian','gaussian','gaussian'],
#                        [['gaussian','gaussian','gaussian'],['gaussian','gaussian','gaussian']],
#                       [['gaussian','gaussian','gaussian']]]

#         theta_cop1 = [[0.7, 0.8, 0.9],
#                       [[0.3, 0.4, 0.5],[0.6, 0.7, 0.8]],
#                       [[0.2, 0.5, 0.9]]]

#         margin_cop1 = [['gaussian','gaussian'],
#                       [['gaussian','gaussian','gaussian']]]

#         theta_cop1 = [[0.7, 0.8],
#                       [[0.2, 0.5, 0.9]]]


    d = len(r_matrix)
    cop_vine = []
    for tr in range(0,d-1,1):
        cop_vine1 = []
        for col in range(0,d-1-tr,1):
            if (tr == 0) | (binning == False):
                cop_p = cop_par_obj(margin_cop1[tr][col],theta_cop1[tr][col])
                cop_vine1.append(cop_p)
            else:
                cop_vine11 = []
                for bb in range(0,n_bin,1):
                    cop_p = cop_par_obj(margin_cop1[tr][col][bb],theta_cop1[tr][col][bb])
                    cop_vine11.append(cop_p)
                cop_vine1.append(cop_vine11)
        cop_vine.append(cop_vine1)

    for tr in range(0,d-1,1):
        for col in range(0,d-1-tr,1):
            if (tr == 0) | (binning == False):
                print('edge: {} '.format(matrix_edges[tr][col]), 'cop family: {}'.format(cop_vine[tr][col].family), 'theta: {}'.format(cop_vine[tr][col].theta))
            else:
                for bb in range(0,n_bin,1):
                    print('edge: {} '.format(matrix_edges[tr][col]),'bin: {} '.format(bb), 'cop family: {}'.format(cop_vine[tr][col][bb].family), 'theta: {}'.format(cop_vine[tr][col][bb].theta))

    ################# IF YOU WANT TO USE C-VINE OR D-VINE    ################################### 
elif (vine_type == 'c-vine') | (vine_type == 'd-vine'):
    
    
    r_matrix, ind_vine, nodes, matrix_edges = prepare_vine(vine_type, dim)

    print(r_matrix)

    binning = False
    n_bin = 3
    
    ########## DEFINE MARGINS
    
    margin_vine = []
    for i in range(0,dim,1):
        mar_p = margin_obj('norm', [0,1], True)
        margin_vine.append(mar_p)

    for i in range(0,dim,1):
        print(margin_vine[i].dist, end =' ')
        print(margin_vine[i].theta, end =' ')

    ############## DEFINE COPULAS
    # NN = 0
    tr = 0
    cop_vine = []
    for i in range(dim,1,-1):
        cop_vine1 = []
        for j in range(0,i-1,1):
            if (tr == 0) | (binning == False):
    #             if tr == NN:
    #                 cop_p = cop_par_obj('ind',[])  #
    #                 cop_vine1.append(cop_p)
    #             else:
                cop_p = cop_par_obj('clayton',4.5)  #
                cop_vine1.append(cop_p)
            else:
                cop_vine11 = []
                for bb in range(0,n_bin,1):
                    cop_p = cop_par_obj('gaussian',0.9)
                    cop_vine11.append(cop_p)
                cop_vine1.append(cop_vine11)
        cop_vine.append(cop_vine1)
        tr += 1
    
#     margin_cop1 = [['gaussian','student','clayton','gaussian'],
#                    ['student','clayton','gaussian'],
#                    ['student','gaussian'],
#                    ['clayton']]
        
#     theta_cop1 = [[0.3, [0,0.2], 0.7,  -0.8],
#                   [[-0.8,0.2], 4.5,  -0.2],
#                   [[0,0.5],  -0.7],
#                   [0.9]]
    
#     d = len(r_matrix)
#     cop_vine = []
#     for tr in range(0,d-1,1):
#         cop_vine1 = []
#         for col in range(0,d-1-tr,1):
#             if (tr == 0) | (binning == False):
#                 cop_p = cop_par_obj(margin_cop1[tr][col],theta_cop1[tr][col])
#                 cop_vine1.append(cop_p)
#             else:
#                 cop_vine11 = []
#                 for bb in range(0,n_bin,1):
#                     cop_p = cop_par_obj(margin_cop1[tr][col][bb],theta_cop1[tr][col][bb])
#                     cop_vine11.append(cop_p)
#                 cop_vine1.append(cop_vine11)
#         cop_vine.append(cop_vine1)

    d = len(r_matrix)
    for tr in range(0,d-1,1):
        for col in range(0,d-1-tr,1):
            if (tr == 0) | (binning == False):
                print('edge: {} '.format(matrix_edges[tr][col]), 'cop family: {}'.format(cop_vine[tr][col].family), 'theta: {}'.format(cop_vine[tr][col].theta))
            else:
                for bb in range(0,n_bin,1):
                    print('edge: {} '.format(matrix_edges[tr][col]),'bin: {} '.format(bb), 'cop family: {}'.format(cop_vine[tr][col][bb].family), 'theta: {}'.format(cop_vine[tr][col][bb].theta))

# if binning == True:
#     exc = tf.math.floormod(cases,n_bin)
#     cases = cases - exc

sample, v, v_flip, tau_corr, tau_bins = generate_r_samples(cases, r_matrix, ind_vine, nodes, margin_vine, cop_vine, n_bin, binning)
print(sample)

nodes:
[ 1  2  3  4  5  6  7  8  9 10]
edges:
['(1,2)', '(1,3)', '(1,4)', '(1,5)', '(1,6)', '(1,7)', '(1,8)', '(1,9)', '(1,10)']
['(2,3|1)', '(2,4|1)', '(2,5|1)', '(2,6|1)', '(2,7|1)', '(2,8|1)', '(2,9|1)', '(2,10|1)']
['(3,4|2,1)', '(3,5|2,1)', '(3,6|2,1)', '(3,7|2,1)', '(3,8|2,1)', '(3,9|2,1)', '(3,10|2,1)']
['(4,5|3,2,1)', '(4,6|3,2,1)', '(4,7|3,2,1)', '(4,8|3,2,1)', '(4,9|3,2,1)', '(4,10|3,2,1)']
['(5,6|4,3,2,1)', '(5,7|4,3,2,1)', '(5,8|4,3,2,1)', '(5,9|4,3,2,1)', '(5,10|4,3,2,1)']
['(6,7|5,4,3,2,1)', '(6,8|5,4,3,2,1)', '(6,9|5,4,3,2,1)', '(6,10|5,4,3,2,1)']
['(7,8|6,5,4,3,2,1)', '(7,9|6,5,4,3,2,1)', '(7,10|6,5,4,3,2,1)']
['(8,9|7,6,5,4,3,2,1)', '(8,10|7,6,5,4,3,2,1)']
['(9,10|8,7,6,5,4,3,2,1)']
[[10  0  0  0  0  0  0  0  0  0]
 [ 9  9  0  0  0  0  0  0  0  0]
 [ 8  8  8  0  0  0  0  0  0  0]
 [ 7  7  7  7  0  0  0  0  0  0]
 [ 6  6  6  6  6  0  0  0  0  0]
 [ 5  5  5  5  5  5  0  0  0  0]
 [ 4  4  4  4  4  4  4  0  0  0]
 [ 3  3  3  3  3  3  3  3  0  0]
 [ 2  2  2  2  2  2  2  2  

ValueError: Memory growth cannot differ between GPU devices

In [ ]:
fig, axs = plt.subplots(dim, dim,figsize=(30,30))
for i in range(0,dim,1):
    for j in range(i+1,dim,1):
#         axs[i,j].plot(x[:,i],x[:,j],'b.')    
        axs[i,j].plot(sample[:,i],sample[:,j],'b.')    
        axs[i,j].set_title(str(i+1)+","+str(j+1))

In [ ]:
load_mat = False
load_pickle = False

if load_mat:
    mat_contents = sio.loadmat('stu_ex_01.mat') #sim_vine #sim17_10000.mat')   #X_sim17.mat')
    # print(mat_contents)
    dat = mat_contents.get('x') #X_sim17
    dat = np.array(dat,np.float64)

    pdf_cop = mat_contents.get('pdf_cop') #X_sim17
    pdf_cop = np.array(pdf_cop,np.float32)
    
if load_pickle:
    pickle_in = open("ex_clay20_1","rb")
    dict_save = pickle.load(pickle_in)
    vine_copulas = dict_save["vine_copulas"]
    x = dict_save["x"]
    r_matrix = dict_save["r_matrix"]
    vine_depth = 20

In [ ]:
################### DEFINE VINE #################

vine_type = "d-vine"
method = 'matrix' #'matrix' 'optimal'
families = "kercop"
knots = 50

vine_depth = len(r_matrix)

# vine_depth = 20 #len(r_matrix)

margin_vine = []
for i in range(0,vine_depth,1):
    mar_p = margin_obj('norm', [0,1], True)
    margin_vine.append(mar_p)
    
vine = vine_obj_bin(vine_type, families, vine_depth, margin_vine, knots, method, r_matrix)

if load_pickle:
    vine.copulas = vine_copulas

### Vine fitting

General parameters:
- parallel: True or False           (Fit in parallel each level)
- binning: True or False            (It can be True only if parallel is False)
- param: True or False              (Parametric or Non-parametric)
- vine_depth: any                   (Max level of vine to fit)

Parametric parameters:
- param_families: ["ind","gaussian","student","clayton","claytonrot90"]   (Decide which parametric families to fit)

Non-parametric parameters:
- opt_method: 'LL1' or 'LL2'

Binning parameters:
- n_bin: any                        (Number of bins)

In [ ]:
param = False
binning = False
n_bin = 3

## Make data divisible for bins and k-fold
# x = dat # sample  #dat
x = sample ## CHANGE THIS IF YOU WANT TO USE DATA THAT ARE LOADED
x = np.array(x,np.float32)


if binning == True:
    if param == False:
        exc = tf.math.floormod(tf.shape(x)[0],n_bin*5)
    else:
        exc = tf.math.floormod(tf.shape(x)[0],n_bin)
    x = x[:tf.shape(x)[0]-exc,:]
else:
    if param == False:
        exc = tf.math.floormod(tf.shape(x)[0],5)
        x = x[:tf.shape(x)[0]-exc,:]

## Prepare copula

sort_n = 'rand'
e = prep_cop(x, vine, sort_n)
print(e)

In [ ]:
### FITTING
# Parameters:
# - Data: x
# - Parallel: True or False
# - Bandwidth optimization: LL1 or LL2
# - Binning: True or false.    It can be True only if parallel is false
# - n_bin: Select the number of bins
# parallel = False

gen_dict = {'parallel':True, 'binning':binning, 'param':param, 'vine_depth':vine_depth, 'fitted':False}  #vine_depth
par_dict = {'param_families':["ind","gaussian"]}  #["ind","gaussian","student","clayton","claytonrot90"]
npc_dict = {'opt_method':'LL1','batch_paral':3}
bin_dict = {'n_bin':n_bin}

save_vine = False

vine.fit(x,gen_dict,npc_dict,par_dict,bin_dict)

if save_vine:
    dict_save = {'vine_copulas': vine.copulas, 'r_matrix': vine.r_matrix, 'data': sample}
    pickle_out = open("clay_20_ale","wb")
    pickle.dump(dict_save,pickle_out)
    pickle_out.close()

In [ ]:
sample, _, _, _ = vine_copula_sample(vine,2000)

In [ ]:
fig, axs = plt.subplots(vine_depth, vine_depth,figsize=(30,30))
for i in range(0,vine_depth,1):
    for j in range(i+1,vine_depth,1):
        axs[i,j].plot(x[:,i],x[:,j],'b.')    
        axs[i,j].plot(sample[:,i],sample[:,j],'r.')    
        axs[i,j].set_title(str(i+1)+","+str(j+1))

#### Vine evaluation

In [ ]:
########### CREATE POINTS FOR EVALUATION

exp_dim = 100
dim = 0

points = create_points(x,dim,exp_dim)
print(points)

p, p_cop, _ = vine.evaluation(points)

In [ ]:
save_vine = False
if save_vine:
    dict_save = {'vine_copulas': vine.copulas, 'r_matrix': vine.r_matrix, 'data': sample}
    pickle_out = open("clay_20_ale","wb")
    pickle.dump(dict_save,pickle_out)
    pickle_out.close()

#### Predict response

In [ ]:
############### PREDICT VINE ##################
dim = 0
exp_dim = 100

p, y_ml, y_em = predict_vine(x,vine,dim,exp_dim)

In [ ]:
print(tf.where(tf.math.is_nan(y_ml)))
print(tf.where(tf.math.is_nan(y_em)))

#### IF NAN 
from utils.tensor_op import replace_nan_inf

y_em = replace_nan_inf(y_em)

## This problem occur only with Clayton with high correlation..
## I checked this and try to solve, but it seems that in the nearest interpolation when data_s try to find the corresponding value
## in the pd_grid_uv it finds a probability of 0. Therefore, the logs give -inf and it results in NaN.
## Initially, I solved this by setting in evalu/vine_eval/vine_fitting this part:
## ker_grid_all = ker_grid_all + 1e-15*NORM adding 1e-15*NORM
## Otherwise it might be a problem in the way the kernel_cdf or inverse_kernel_cdf or any interpolation set the boundary.
## I tried to check this but it requires major investigation...
## This occurs only when there are very very high correlation as in the clayton case with high vine dimension.

In [ ]:
from scipy import stats
corr = stats.pearsonr(x[:,dim], y_em)

print(corr[0])

plt.figure()
plt.plot(x[:,dim], y_ml, 'r.')
plt.plot(x[:,dim], y_em, 'b.')
plt.title('Correlation: ' + str(corr[0]))
plt.show()

plt.figure()
plt.plot(y_em, x[:,0], 'g.')
plt.plot(x[:,dim], x[:,0], 'y.')
plt.show()

###############

plt.figure()
plt.plot(y_em, x[:,1], 'g.')
plt.plot(x[:,dim], x[:,1], 'y.')
plt.show()

###############

plt.figure()
plt.plot(y_em, x[:,2], 'g.')
plt.plot(x[:,dim], x[:,2], 'y.')
plt.show()



#### Vine entropy estimation

In [ ]:
info_dict = {'cases':1000, 'iterations':10, 'alpha': 0.05}
MI = vine_entropy(vine,info_dict)
print(MI)